# **TUTORIAL**: Study objective B0, dynamic CC (deterministic)

AESA Phase B: (dynamic) carrying capacities definition.\
Extract AR6 climate change pathways from `process_ar6(...)` outputs.

# Before starting...

### Set workspace

Run `set_workspace(...)` before any later function call. It sets the active workspace root
for the current Python session, creates or reuses the workspace, imports package
prerequisites under `data_raw/`, and records setup guidance in `data_raw/summary.log`.

In [ ]:
from pyaesa import set_workspace

# Windows example; update this path before running.
set_workspace(r"C:\Users\username\Documents\aesa_workspace")

# macOS example; update this path before running.
# set_workspace("/Users/username/Documents/aesa_workspace")

### Run prerequisites

This tutorial assumes that the workspace has already been set, with all prerequisites available (downloads and processing).\
If this is not your case, it is recommend to first go through all notebooks available in the `tutorials/core_prerequisites` folder, to have extensive details on what the functions do.

# Basic features of the deterministic function

### In a nutshell

The function takes necessary **inputs**:
-  `years` (must contain at least two consecutive years).
- `category` and `ssp_scenario` restrict the retained AR6 pathway pool.

The deterministic **output** folder contains:
- dynamic AR6 carrying capacity tables for the
selected emission variable and years.
- **Figures** are rendered when requested, as controlled with the `figures` and `figure_format` parameters.
- log files

Basic features also involve:
- **overwriting** of existing values: use the `refresh` parameter.

Methodological notes available in
`methodological_notes/` are imported with package
prerequisites:

* `data_raw/methodological_notes/` contains notes for definition
  of functional units and allocation methods, prospective allocation, and
  uncertainty sources.
* `data_raw/carrying_capacities/` contains the note for definition of steady
  state and dynamic carrying capacities. This explains AR6
scenario filtering, harmonization, and gross emissions modes.

### Public argument checklist
The table lists all arguments; the same definitions are available in the function docstring.

<div class="pyaesa-argument-legend">
<div class="pyaesa-default-block" style="color:#087f5b"><strong>Green items = default if omitted.</strong></div>
<div class="pyaesa-optional-block" style="color:#c45f00"><strong>Orange items = optional feature skipped if omitted.</strong></div>
</div>

Do not write green or orange items when that behavior is intended.

<details open>
<summary><code>deterministic_ar6_cc(...)</code> arguments</summary>

<table>
<thead><tr><th>Argument</th><th>Description</th></tr></thead>
<tbody>
<tr><td style="vertical-align:top; white-space:nowrap;"><code>years</code></td><td>Study year selector provided as a consecutive year<br>&#10;list or <code>range(start_year, end_year + 1)</code>. The resolved years<br>&#10;must contain at least two consecutive years with no gaps.</td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>harmonization</code></td><td>Whether to harmonize retained AR6 pathways to the<br>&#10;historical baseline. <strong>Defaults to</strong> <code>True</code>.</td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>harmonization_method</code></td><td>Harmonization method applied only when<br>&#10;<code>harmonization=True</code>. <strong>Defaults to</strong> <code>&quot;offset&quot;</code>. The only<br>&#10;supported value is currently <code>&quot;offset&quot;</code>.<br>&#10;Ignored when <code>harmonization=False</code>.</td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>category</code></td><td>AR6 category classification filter for global warming<br>&#10;trajectories. <strong>Defaults to</strong> <code>[&quot;C1&quot;, &quot;C2&quot;, &quot;C3&quot;, &quot;C4&quot;]</code>. Pass a<br>&#10;string such as <code>&quot;C2&quot;</code> or a list such as <code>[&quot;C1&quot;, &quot;C2&quot;]</code> to<br>&#10;restrict.</td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>ssp_scenario</code></td><td>SSP scenario filter. <strong>Defaults to</strong><br>&#10;<code>[&quot;SSP1&quot;, &quot;SSP2&quot;, &quot;SSP3&quot;, &quot;SSP4&quot;, &quot;SSP5&quot;]</code>. Pass a string<br>&#10;such as <code>&quot;SSP2&quot;</code> or a list such as <code>[&quot;SSP1&quot;, &quot;SSP2&quot;]</code> to<br>&#10;restrict.</td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>emission_type</code></td><td>Dynamic AR6 emission type. Accepted values are<br>&#10;<code>&quot;kyoto_gases&quot;</code> (<strong>default</strong>) and <code>&quot;co2&quot;</code>.<br>&#10;<code>emission_type=&quot;kyoto_gases&quot;</code> uses the GWP100 Kyoto Gases<br>&#10;aggregate; <code>emission_type=&quot;co2&quot;</code> uses direct CO2 pathways.</td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>include_afolu</code></td><td>Whether AFOLU is included inside the selected<br>&#10;<code>emission_type</code>. <code>False</code> uses the <code>WO AFOLU</code> pathway<br>&#10;family. <code>True</code> uses the AFOLU-inclusive family. <strong>Defaults to</strong><br>&#10;<code>False</code>.</td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>emissions_mode</code></td><td>Dynamic AR6 emissions mode. Accepted values are<br>&#10;<code>&quot;net&quot;</code>, <code>&quot;gross&quot;</code>, and <code>&quot;gross_alt&quot;</code>. <strong>Defaults to</strong><br>&#10;<code>&quot;gross_alt&quot;</code>. <code>emissions_mode</code> selects net, gross, or gross<br>&#10;alternative emissions. <code>&quot;gross&quot;</code> removes all sequestration<br>&#10;sources from net emissions. <code>&quot;gross_alt&quot;</code> removes all<br>&#10;sequestration sources except CCS, as it does not directly capture<br>&#10;CO2 from the atmosphere; IPCC AR6 recommends treating CCS<br>&#10;separately from net negative sequestration. Gross modes write<br>&#10;positive emissions denominator rows and signed negative<br>&#10;sequestration companion rows; downstream allocated carrying<br>&#10;capacity (aCC) and absolute sustainability ratio (ASR) consume<br>&#10;only the denominator gross positive rows. See<br>&#10;<code>data_raw/methodological_notes/methodological_note__steady_state__dynamic_cc.pdf</code><br>&#10;for the methodological explanation.</td></tr>
<tr class="pyaesa-optional-row" style="color:#c45f00;"><td style="vertical-align:top; white-space:nowrap;"><code>subset_version</code></td><td>Optional selector for a subset of AR6 model-scenario<br>&#10;pairs. Follow<br>&#10;<code>data_processed/ar6/&lt;processed_scope&gt;/README_model_scenario_subset.txt</code><br>&#10;to create the subset CSV. <strong>Omit</strong> this argument to use every retained<br>&#10;model-scenario pair.<br>&#10;<strong>Defaults to</strong> <code>None</code>.</td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>output_format</code></td><td>Persisted output file format: <code>&quot;csv&quot;</code> (<strong>default</strong>),<br>&#10;<code>&quot;pickle&quot;</code>, or <code>&quot;parquet&quot;</code>.</td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>figures</code></td><td>Whether to render figures.<br>&#10;<strong>Default is</strong> <code>True</code>.</td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>figure_format</code></td><td>Figure render settings mapping. <strong>Defaults to</strong><br>&#10;<code>{&quot;format&quot;: &quot;png&quot;, &quot;dpi&quot;: 500}</code>.<br>&#10;&#160;<br>&#10;Nested keys:<br>&#10;&#160;<br>&#10;&bull; <code>format</code>: Figure file format. Accepted values are <code>&quot;png&quot;</code>,<br>&#10;  <code>&quot;pdf&quot;</code>, and <code>&quot;svg&quot;</code>.<br>&#10;&bull; <code>dpi</code>: Positive integer figure resolution used for raster<br>&#10;  outputs.</td></tr>
<tr class="pyaesa-default-row" style="color:#087f5b;"><td style="vertical-align:top; white-space:nowrap;"><code>refresh</code></td><td>If <code>True</code>, clear and recompute the resolved deterministic AR6 CC output scope for the requested study period, harmonization flag, harmonization method, emission variables, and subset version, plus the matching processed AR6 output scope selected by <code>process_ar6(...)</code> for that request. The cleared AR6 CC scope is the selector-specific <code>ar6_cc</code> deterministic output folder beside that processed AR6 scope. For example, for years 2019 to 2060, default harmonization, default Kyoto gas settings, category <code>[&quot;C1&quot;]</code>, and SSP <code>[&quot;SSP1&quot;]</code>, the refreshed path is <code>&lt;repo&gt;/data_processed/ar6/2019-2060_harmonization_offset/ar6_cc/gross_alt_kyoto_gases_wo_afolu/C1__SSP1/deterministic</code>. Raw downloads and downstream aCC or ASR outputs are not refreshed. <strong>Defaults to</strong> <code>False</code>.</td></tr>
</tbody>
</table>

</details>


### Extract dynamic AR6 CC for all AR6 categories and SSPs, using defaults where omitted

In [ ]:
from pyaesa import deterministic_ar6_cc

deterministic_ar6_cc(
    years=range(2020, 2051),
)

### GHG (Kyoto gases) or CO2 emissions

In [ ]:
deterministic_ar6_cc(
    years=range(2020, 2051),
    category=["C1", "C2"],
    ssp_scenario=["SSP2"],
    emission_type="co2",
)

# What to do next

**Beginners**

If you are a new user in the process of discovering <span style="color:#366e9c"><strong><tt>py</tt></strong></span><span style="color:#c83737"><strong><tt>aesa</tt></strong></span>, it is recommend that you first discover all study objectives with the **basic features** available.
Have a look at the other notebooks available at `tutorials/study_objectives/...`

**Experts**

If you are already familiar with <span style="color:#366e9c"><strong><tt>py</tt></strong></span><span style="color:#c83737"><strong><tt>aesa</tt></strong></span> and if you want to discover **advanced features** available, check out examples in the next section of this tutorial !

# Advanced features

Advanced features currently available includes:
- Subset of model-scenarios
- Net, gross, and gross_alt emissions
- Including or excluding AFOLU emissions
- Custom carrying capacities definition
- Changing the emissions harmonization method (not yet available!)

### Subset of model-scenarios

`subset_version` selects a custom model scenario pair subset prepared from `data_processed/ar6/<processed_scope>/README_model_scenario_subset.txt`.

In [ ]:
deterministic_ar6_cc(
    years=range(2020, 2051),
    category=["C1", "C2"],
    ssp_scenario=["SSP2"],
    subset_version="selected_pairs",
)

### Net, gross, and gross_alt emissions

`emission_mode`

<p style="color:#b00020"><strong>IN PROGRESS:</strong> Example available shortly! </p>

### Including or excluding AFOLU emissions

`include_afolu`

<p style="color:#b00020"><strong>IN PROGRESS:</strong> Example available shortly! </p>

### Custom carrying capacities definition

For custom static carrying capacity references, use the CSV templates in
`data_raw/carrying_capacities/`. Details instructions are available in
`data_raw/carrying_capacities/README_add_custom_carrying_capacities.txt`.

For dynamic AR6 routes, `subset_version` selects a custom model scenario
pair subset prepared from
`data_processed/ar6/<processed_scope>/README_model_scenario_subset.txt`.